# Konuşmadan Duygu Tanıma: Colab'de baştan sona çalıştırma

Çalışma zamanı türünü **GPU (T4)** seçin. Tüm hat GPU'da yaklaşık 1-2 saat sürer.
Proje zip'ini (`ses-duygu-tanima.zip`) sol taraftaki dosya paneline yükleyin.

In [ ]:
!unzip -q -o ses-duygu-tanima.zip -d /content/ser && cd /content/ser && pip install -q -r requirements.txt

## 1. Veri setlerini resmi kaynaklarından indir

In [ ]:
%cd /content/ser
!mkdir -p data/raw/ravdess data/raw/emodb
!wget -q -O /tmp/ravdess.zip 'https://zenodo.org/records/1188976/files/Audio_Speech_Actors_01-24.zip?download=1'
!unzip -q -o /tmp/ravdess.zip -d data/raw/ravdess
!wget -q -O /tmp/emodb.zip 'http://www.emodb.bilderbar.info/download/download.zip'
!unzip -q -o /tmp/emodb.zip -d data/raw/emodb
!find data/raw/ravdess -name '*.wav' | wc -l   # 1440 olmalı
!find data/raw/emodb -name '*.wav' | wc -l     # 535 olmalı

## 2. Hazırlık ve bölmeler

In [ ]:
!python src/prepare_data.py && python src/make_splits.py

## 3. WavLM-Large özellikleri (Hugging Face'ten indirilir)

In [ ]:
!python src/ssl_extract.py --model microsoft/wavlm-large --tag wavlm_large --variants clean snr10 snr0 aug1 --cache_layers 22

## 4. Deneyler

In [ ]:
!python src/baseline_lstm.py --exp ravdess8_spk --save_model
!python src/baseline_lstm.py --exp ravdess8_rnd
!python src/baseline_lstm.py --exp ravdess8_spk --augment
!for s in R E RE; do python src/baseline_lstm.py --exp cross6_$s; done

In [ ]:
!python src/ssl_heads.py --exp ravdess8_spk --mode probe
!for e in ravdess8_spk ravdess8_rnd cross6_R cross6_E cross6_RE; do python src/ssl_heads.py --exp $e; done
!python src/ssl_heads.py --exp ravdess8_spk --augment

In [ ]:
!python src/finetune_ssl.py --exp ravdess8_spk --model microsoft/wavlm-large --cache_layer 22 --n_layers 2 --epochs 10 --patience 3

## 5. Sonuçlar ve demo

In [ ]:
!python src/analyze_results.py
from IPython.display import Image, Markdown
display(Markdown(open('results/summary.md').read()))
display(Image('report/figures/fig_main_comparison.png'))

In [ ]:
!python src/train_demo_model.py
!python src/app_gradio.py --backbone microsoft/wavlm-large --share